#SAM3 auto-labeling sample video

In [ ]:
import os
import json
import cv2
import numpy as np
import torch
from PIL import Image
from transformers import Sam3Processor, Sam3Model


# =========================
# CONFIG
# =========================

VIDEO_PATH      = "/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4"
FRAME_STRIDE    = 10                   # Lấy mỗi N frames
START_FRAME     = 0
MAX_FRAMES      = -1                    # -1 = toàn bộ

# --- SAM3 ---
SAM3_MODEL_ID   = "facebook/sam3"
SAM3_TEXT_PROMPT = "vehicle"
SAM3_THRESH     = 0.5
SAM3_MASK_THRESH = 0.5
SAM3_BOX_SCORE_THR = 0.5
SAM3_DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
SAM3_USE_AMP    = (SAM3_DEVICE == "cuda")

# --- Output ---
OUTPUT_DIR      = "./output_sam3_0-5-conf-exclude-signal-10fps"
SAVE_JSON       = True

# --- Crop ---
CROP_MARGIN     = 0.05

# --- Vùng loại trừ (biển báo có hình xe) ---
# Bbox nào có tâm nằm trong vùng này sẽ bị bỏ hoàn toàn
# Format: list of [x1, y1, x2, y2] pixel, margin tự mở rộng thêm
EXCLUDE_ZONES = [
    [1040, 260, 1150, 360],             # Vùng biển báo có hình ô tô
]
EXCLUDE_ZONE_MARGIN = 30               # Mở rộng thêm mỗi chiều (pixel)

# --- Panel layout ---
PANEL_W         = 640
PANEL_SEPARATOR = 12
PANEL_BG        = (0, 0, 0)
STAGE_COLS      = 4
TILE_PAD        = 10
TITLE_H         = 46
TITLE_FONT_SCALE = 0.95
TITLE_THICK     = 2


# =========================
# OUTPUT FOLDERS
# =========================
FRAMES_ORIG     = os.path.join(OUTPUT_DIR, "frames_original")
FRAMES_BOXES    = os.path.join(OUTPUT_DIR, "frames_sam3_boxes")
CUTOUTS         = os.path.join(OUTPUT_DIR, "cutouts")
FRAMES_PANEL    = os.path.join(OUTPUT_DIR, "frames_panel")
JSON_DIR        = os.path.join(OUTPUT_DIR, "json_per_frame")
LABELS_TXT      = os.path.join(OUTPUT_DIR, "labels_txt")
LABELS_XML      = os.path.join(OUTPUT_DIR, "labels_xml")

for d in [FRAMES_ORIG, FRAMES_BOXES, CUTOUTS, FRAMES_PANEL, JSON_DIR, LABELS_TXT, LABELS_XML]:
    os.makedirs(d, exist_ok=True)


# =========================
# HELPERS
# =========================
def draw_label_box(img, x1, y1, text, color, font_scale, txt_thickness, pad_x, pad_y, gap_y):
    (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, txt_thickness)
    y_text_bottom = y1 - gap_y
    if y_text_bottom - th - 2 * pad_y < 0:
        y_text_bottom = y1 + th + 2 * pad_y + gap_y

    bg_tl = (x1, y_text_bottom - th - 2 * pad_y)
    bg_br = (x1 + tw + 2 * pad_x, y_text_bottom + baseline)
    cv2.rectangle(img, bg_tl, bg_br, color, thickness=-1)

    text_org = (x1 + pad_x, y_text_bottom - pad_y)
    cv2.putText(img, text, text_org, cv2.FONT_HERSHEY_SIMPLEX,
                font_scale, (0, 0, 0), txt_thickness, cv2.LINE_AA)


def resize_pad(img_bgr, out_w, out_h, bg=(0, 0, 0)):
    out_w = int(max(1, out_w))
    out_h = int(max(1, out_h))

    if img_bgr is None or not isinstance(img_bgr, np.ndarray) or img_bgr.size == 0:
        return np.full((out_h, out_w, 3), bg, dtype=np.uint8)

    h, w = img_bgr.shape[:2]
    if h <= 0 or w <= 0:
        return np.full((out_h, out_w, 3), bg, dtype=np.uint8)

    scale = min(out_w / w, out_h / h)
    nw = int(max(1, round(w * scale)))
    nh = int(max(1, round(h * scale)))

    resized = cv2.resize(img_bgr, (nw, nh), interpolation=cv2.INTER_AREA)
    out = np.full((out_h, out_w, 3), bg, dtype=np.uint8)

    x0 = (out_w - nw) // 2
    y0 = (out_h - nh) // 2
    out[y0:y0+nh, x0:x0+nw] = resized
    return out


def build_crop_panel(frame_h, items):
    """Tạo panel bên phải hiển thị các crops."""
    panel = np.full((frame_h, PANEL_W, 3), PANEL_BG, dtype=np.uint8)

    # Title
    title = f"SAM3 Crops: {SAM3_TEXT_PROMPT} ({len(items)} found)"
    cv2.putText(panel, title, (12, int(TITLE_H * 0.72)),
                cv2.FONT_HERSHEY_SIMPLEX, TITLE_FONT_SCALE,
                (255, 255, 255), TITLE_THICK, cv2.LINE_AA)
    cv2.line(panel, (10, TITLE_H - 8), (PANEL_W - 10, TITLE_H - 8),
             (120, 120, 120), 2, cv2.LINE_AA)

    n = len(items)
    if n == 0:
        cv2.putText(panel, "No detections", (14, TITLE_H + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (200, 200, 200), 2, cv2.LINE_AA)
        return panel

    cols = int(max(1, STAGE_COLS))
    grid_y = TITLE_H
    grid_h = max(10, frame_h - TITLE_H)

    tile_w = (PANEL_W - (cols + 1) * TILE_PAD) // cols
    tile_w = int(max(40, tile_w))

    rows = int(np.ceil(n / cols))
    rows = int(max(1, rows))

    tile_h = (grid_h - (rows + 1) * TILE_PAD) // rows
    tile_h = int(max(40, tile_h))

    for i, it in enumerate(items):
        r = i // cols
        c = i % cols
        tx = TILE_PAD + c * (tile_w + TILE_PAD)
        ty = grid_y + TILE_PAD + r * (tile_h + TILE_PAD)
        if ty + tile_h > frame_h:
            break

        tile = resize_pad(it["img"], tile_w, tile_h, bg=PANEL_BG)
        panel[ty:ty+tile_h, tx:tx+tile_w] = tile

        bcol = it.get("color", (0, 200, 255))
        cv2.rectangle(panel, (tx, ty), (tx + tile_w, ty + tile_h), bcol, 2)

        # Label
        text = it.get("text", "")
        if text:
            bar_h = max(22, int(tile_h * 0.18))
            cv2.rectangle(panel, (tx, ty + tile_h - bar_h), (tx + tile_w, ty + tile_h), (0, 0, 0), -1)
            cv2.putText(panel, text[:28], (tx + 6, ty + tile_h - 7),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

    return panel


# =========================
# SAVE ANNOTATIONS
# =========================
def save_annotation_txt(filepath, img_shape, detections):
    """
    Lưu YOLO format: class_id x_center y_center width height
    Tọa độ normalize về [0, 1]. Class 0 = SAM3_TEXT_PROMPT.
    """
    h, w = img_shape[:2]
    with open(filepath, "w") as f:
        for det in detections:
            x1, y1, x2, y2 = det["bbox_xyxy"]
            xc = max(0, min(1, ((x1 + x2) / 2.0) / w))
            yc = max(0, min(1, ((y1 + y2) / 2.0) / h))
            bw = max(0, min(1, (x2 - x1) / w))
            bh = max(0, min(1, (y2 - y1) / h))
            f.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")


def save_annotation_xml(filepath, image_filename, img_shape, detections):
    """Lưu Pascal VOC XML format."""
    import xml.etree.ElementTree as ET
    from xml.dom import minidom

    h, w, c = img_shape

    annotation = ET.Element("annotation")
    ET.SubElement(annotation, "folder").text = "images"
    ET.SubElement(annotation, "filename").text = image_filename

    size = ET.SubElement(annotation, "size")
    ET.SubElement(size, "width").text = str(w)
    ET.SubElement(size, "height").text = str(h)
    ET.SubElement(size, "depth").text = str(c)
    ET.SubElement(annotation, "segmented").text = "0"

    for det in detections:
        obj = ET.SubElement(annotation, "object")
        ET.SubElement(obj, "name").text = SAM3_TEXT_PROMPT
        ET.SubElement(obj, "pose").text = "Unspecified"
        ET.SubElement(obj, "truncated").text = "0"
        ET.SubElement(obj, "difficult").text = "0"
        ET.SubElement(obj, "confidence").text = f"{det['score']:.4f}"

        bndbox = ET.SubElement(obj, "bndbox")
        x1, y1, x2, y2 = det["bbox_xyxy"]
        ET.SubElement(bndbox, "xmin").text = str(int(x1))
        ET.SubElement(bndbox, "ymin").text = str(int(y1))
        ET.SubElement(bndbox, "xmax").text = str(int(x2))
        ET.SubElement(bndbox, "ymax").text = str(int(y2))

    xml_str = minidom.parseString(ET.tostring(annotation)).toprettyxml(indent="  ")
    lines = xml_str.split("\n")
    xml_str = "\n".join(lines[1:])

    with open(filepath, "w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        f.write(xml_str)


# =========================
# SAM3 DETECT
# =========================
def sam3_detect_boxes(img_bgr, text_prompt):
    """Detect objects bằng SAM3 text prompt, trả về boxes và scores."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(img_rgb)

    inputs = sam3_processor(images=pil, text=text_prompt, return_tensors="pt")
    inputs = {k: v.to(SAM3_DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        if SAM3_USE_AMP:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = sam3_model(**inputs)
        else:
            outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs,
        threshold=SAM3_THRESH,
        mask_threshold=SAM3_MASK_THRESH,
        target_sizes=inputs["original_sizes"].tolist()
    )[0]

    boxes = results.get("boxes", None)
    scores = results.get("scores", None)

    if boxes is None or len(boxes) == 0:
        return np.zeros((0, 4), dtype=np.int32), np.zeros((0,), dtype=np.float32)

    boxes_np = np.round(boxes.detach().cpu().numpy()).astype(np.int32)

    if scores is None:
        scores_np = np.ones((len(boxes_np),), dtype=np.float32)
    else:
        scores_np = scores.detach().cpu().numpy().astype(np.float32)

    keep = scores_np >= float(SAM3_BOX_SCORE_THR)
    boxes_np = boxes_np[keep]
    scores_np = scores_np[keep]

    # Lọc bỏ bbox nằm trong vùng loại trừ (biển báo có hình xe)
    if len(boxes_np) > 0 and EXCLUDE_ZONES:
        zone_keep = np.ones(len(boxes_np), dtype=bool)

        for zone in EXCLUDE_ZONES:
            zx1 = zone[0] - EXCLUDE_ZONE_MARGIN
            zy1 = zone[1] - EXCLUDE_ZONE_MARGIN
            zx2 = zone[2] + EXCLUDE_ZONE_MARGIN
            zy2 = zone[3] + EXCLUDE_ZONE_MARGIN

            # Tâm của mỗi bbox
            cx = (boxes_np[:, 0] + boxes_np[:, 2]) / 2.0
            cy = (boxes_np[:, 1] + boxes_np[:, 3]) / 2.0

            in_zone = (cx >= zx1) & (cx <= zx2) & (cy >= zy1) & (cy <= zy2)
            zone_keep &= ~in_zone

        removed = (~zone_keep).sum()
        if removed > 0:
            print(f"    [FILTER] Bỏ {removed} bbox trong vùng biển báo")

        boxes_np = boxes_np[zone_keep]
        scores_np = scores_np[zone_keep]

    return boxes_np, scores_np


# =========================
# PROCESS ONE FRAME
# =========================
def process_frame(img_bgr, img_id):
    H, W = img_bgr.shape[:2]

    font_scale = max(0.45, min(2.2, W / 900.0))
    txt_thickness = max(1, int(round(W / 900.0)))
    box_thickness = max(2, int(round(W / 600.0)))
    pad_x = max(4, int(round(W * 0.004)))
    pad_y = max(3, int(round(W * 0.003)))
    gap_y = max(6, int(round(W * 0.006)))

    # Detect
    boxes_xyxy, scores = sam3_detect_boxes(img_bgr, SAM3_TEXT_PROMPT)

    # Ảnh có boxes
    boxes_img = img_bgr.copy()
    left_img = img_bgr.copy()
    cv2.putText(left_img, img_id, (20, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3, cv2.LINE_AA)

    annos = []
    crop_items = []

    if boxes_xyxy is None or len(boxes_xyxy) == 0:
        panel = build_crop_panel(H, crop_items)
        combined = np.full((H, W + PANEL_SEPARATOR + PANEL_W, 3), PANEL_BG, dtype=np.uint8)
        combined[:, :W] = left_img
        combined[:, W + PANEL_SEPARATOR:] = panel
        return boxes_img, combined, annos

    sam_color = (255, 0, 0)

    for det_i, (box, score) in enumerate(zip(boxes_xyxy, scores), start=1):
        x1, y1, x2, y2 = map(int, box)
        x1 = max(0, min(W - 1, x1))
        y1 = max(0, min(H - 1, y1))
        x2 = max(0, min(W, x2))
        y2 = max(0, min(H, y2))
        if x2 <= x1 or y2 <= y1:
            continue

        # Vẽ box lên boxes_img
        cv2.rectangle(boxes_img, (x1, y1), (x2, y2), sam_color, box_thickness)

        # Vẽ box lên left_img (chỉ box, không label)
        cv2.rectangle(left_img, (x1, y1), (x2, y2), sam_color, box_thickness)

        # Crop với margin
        bw = max(1, x2 - x1)
        bh = max(1, y2 - y1)
        dx = int(bw * CROP_MARGIN)
        dy = int(bh * CROP_MARGIN)
        cx1 = max(0, x1 - dx)
        cy1 = max(0, y1 - dy)
        cx2 = min(W, x2 + dx)
        cy2 = min(H, y2 + dy)

        crop = img_bgr[cy1:cy2, cx1:cx2]
        if crop is None or crop.size == 0:
            continue

        # Lưu crop
        crop_name = f"{img_id}_crop{det_i:02d}.png"
        crop_path = os.path.join(CUTOUTS, crop_name)
        cv2.imwrite(crop_path, crop)

        ch, cw = crop.shape[:2]
        crop_items.append({
            "img": crop,
            "text": f"#{det_i} {score:.2f}",
            "color": (0, 200, 255),
        })

        annos.append({
            "image_id": img_id,
            "det_index": int(det_i),
            "model": SAM3_MODEL_ID,
            "prompt": SAM3_TEXT_PROMPT,
            "score": float(score),
            "bbox_xyxy": [int(x1), int(y1), int(x2), int(y2)],
            "crop_bbox_xyxy": [int(cx1), int(cy1), int(cx2), int(cy2)],
            "crop_path": crop_path,
        })

    # Ghép panel
    panel = build_crop_panel(H, crop_items)
    combined = np.full((H, W + PANEL_SEPARATOR + PANEL_W, 3), PANEL_BG, dtype=np.uint8)
    combined[:, :W] = left_img
    combined[:, W + PANEL_SEPARATOR:] = panel

    return boxes_img, combined, annos


# =========================
# LOAD MODEL
# =========================
print(f"[INFO] Loading SAM3: {SAM3_MODEL_ID} | device: {SAM3_DEVICE}")
sam3_processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID)
sam3_model = Sam3Model.from_pretrained(SAM3_MODEL_ID).to(SAM3_DEVICE)
sam3_model.eval()
print("[INFO] SAM3 loaded!")


# =========================
# RUN ON VIDEO
# =========================
if not os.path.isfile(VIDEO_PATH):
    raise RuntimeError(f"Video not found: {VIDEO_PATH}")

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
base = os.path.splitext(os.path.basename(VIDEO_PATH))[0]

print("=" * 60)
print("  SAM3 VIDEO DETECTOR")
print("=" * 60)
print(f"  Video:       {VIDEO_PATH}")
print(f"  FPS:         {fps:.1f}")
print(f"  Frames:      {total}")
print(f"  Stride:      {FRAME_STRIDE}")
print(f"  Prompt:      '{SAM3_TEXT_PROMPT}'")
print(f"  Threshold:   {SAM3_THRESH}")
print(f"  Device:      {SAM3_DEVICE}")
print(f"  Output:      {OUTPUT_DIR}")
print("=" * 60)

if START_FRAME > 0:
    cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

saved_count = 0
total_dets = 0
frame_idx = START_FRAME

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if (frame_idx - START_FRAME) % FRAME_STRIDE != 0:
        frame_idx += 1
        continue

    if MAX_FRAMES > 0 and saved_count >= MAX_FRAMES:
        break

    t_sec = (frame_idx / fps) if fps and fps > 0 else 0.0
    img_id = f"{base}_f{frame_idx:06d}_t{t_sec:08.2f}".replace(".", "p")

    print(f"[{saved_count + 1}] Frame {frame_idx} | t={t_sec:.2f}s | {img_id}")

    # Lưu frame gốc
    cv2.imwrite(os.path.join(FRAMES_ORIG, f"{img_id}.jpg"), frame)

    # Process
    boxes_img, panel_img, annos = process_frame(frame, img_id)

    # Lưu ảnh boxes
    cv2.imwrite(os.path.join(FRAMES_BOXES, f"{img_id}_boxes.jpg"), boxes_img)

    # Lưu panel
    cv2.imwrite(os.path.join(FRAMES_PANEL, f"{img_id}_panel.jpg"), panel_img)

    # Lưu annotations TXT (YOLO format) và XML (Pascal VOC)
    if annos:
        save_annotation_txt(
            os.path.join(LABELS_TXT, f"{img_id}.txt"),
            frame.shape, annos
        )
        save_annotation_xml(
            os.path.join(LABELS_XML, f"{img_id}.xml"),
            f"{img_id}.jpg", frame.shape, annos
        )

    # Lưu JSON
    if SAVE_JSON:
        with open(os.path.join(JSON_DIR, f"{img_id}.json"), "w", encoding="utf-8") as f:
            json.dump({
                "video": os.path.basename(VIDEO_PATH),
                "frame_index": int(frame_idx),
                "timestamp_sec": float(t_sec),
                "image_id": img_id,
                "model": SAM3_MODEL_ID,
                "prompt": SAM3_TEXT_PROMPT,
                "threshold": SAM3_THRESH,
                "num_detections": len(annos),
                "detections": annos,
            }, f, ensure_ascii=False, indent=2)

    total_dets += len(annos)
    print(f"     -> {len(annos)} detections")

    saved_count += 1
    frame_idx += 1

cap.release()

# Lưu classes.txt
with open(os.path.join(OUTPUT_DIR, "classes.txt"), "w") as f:
    f.write(f"{SAM3_TEXT_PROMPT}\n")

print(f"\n{'=' * 60}")
print(f"  DONE!")
print(f"{'=' * 60}")
print(f"  Frames processed:  {saved_count}")
print(f"  Total detections:  {total_dets}")
print(f"  Output: {OUTPUT_DIR}/")
print(f"     ├── frames_original/")
print(f"     ├── frames_sam3_boxes/")
print(f"     ├── cutouts/")
print(f"     ├── frames_panel/")
print(f"     ├── labels_txt/        (YOLO format for training)")
print(f"     ├── labels_xml/        (Pascal VOC format)")
print(f"     ├── json_per_frame/")
print(f"     └── classes.txt")
print("=" * 60)

[INFO] Loading SAM3: facebook/sam3 | device: cuda


Loading weights: 100%|██████████| 1468/1468 [00:00<00:00, 2745.45it/s]


[INFO] SAM3 loaded!
  SAM3 VIDEO DETECTOR
  Video:       /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4
  FPS:         30.0
  Frames:      6023
  Stride:      10
  Prompt:      'vehicle'
  Threshold:   0.5
  Device:      cuda
  Output:      ./output_sam3_0-5-conf-exclude-signal-10fps
[1] Frame 0 | t=0.00s | 2026-04-15 11-00-45_f000000_t00000p00
    [FILTER] Bỏ 1 bbox trong vùng biển báo
     -> 24 detections
[2] Frame 10 | t=0.33s | 2026-04-15 11-00-45_f000010_t00000p33
    [FILTER] Bỏ 1 bbox trong vùng biển báo
     -> 27 detections
[3] Frame 20 | t=0.67s | 2026-04-15 11-00-45_f000020_t00000p67
    [FILTER] Bỏ 1 bbox trong vùng biển báo
     -> 25 detections
[4] Frame 30 | t=1.00s | 2026-04-15 11-00-45_f000030_t00001p00
    [FILTER] Bỏ 1 bbox trong vùng biển báo
     -> 28 detections
[5] Frame 40 | t=1.33s | 2026-04-15 11-00-45_f000040_t00001p33
    [FILTER] Bỏ 1 bbox trong vùng biển báo
     -> 26 detections
[6] Frame 50 | t=1.67s | 2026

In [ ]:
"""
Dataset Splitter for YOLO Training
"""

import os
import sys
import shutil
import random
import glob
import yaml


# ============================================================
#  CONFIG
# ============================================================

IMAGES_DIR      = "/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/output_sam3_0-5-conf-exclude-signal-10fps/frames_original"     # Thư mục chứa ảnh gốc
LABELS_DIR      = "/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/output_sam3_0-5-conf-exclude-signal-10fps/labels_txt"           # Thư mục chứa labels YOLO txt
OUTPUT_DIR      = "./our-dataset-sam3-10fps-0-5-conf/"                          # Thư mục output dataset mới

TRAIN_RATIO     = 0.7                                # 70% train
VALID_RATIO     = 0.2                                # 20% valid
TEST_RATIO      = 0.1                                # 10% test

CLASS_NAMES     = ["vehicle"]                        # Danh sách tên class
SEED            = 42                                 # Random seed


# ============================================================
#  MAIN
# ============================================================
def main():
    print("=" * 60)
    print("  DATASET SPLITTER (YOLO FORMAT)")
    print("=" * 60)

    # --- Kiểm tra input ---
    if not os.path.isdir(IMAGES_DIR):
        print(f"[ERROR] Images dir not found: {IMAGES_DIR}")
        sys.exit(1)
    if not os.path.isdir(LABELS_DIR):
        print(f"[ERROR] Labels dir not found: {LABELS_DIR}")
        sys.exit(1)

    # --- Tìm tất cả cặp image + label ---
    img_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    all_images = sorted([
        f for f in os.listdir(IMAGES_DIR)
        if f.lower().endswith(img_extensions)
    ])

    # Chỉ giữ ảnh có label tương ứng
    paired = []
    skipped = 0
    for img_name in all_images:
        base = os.path.splitext(img_name)[0]
        label_name = base + ".txt"
        label_path = os.path.join(LABELS_DIR, label_name)

        if os.path.isfile(label_path):
            paired.append((img_name, label_name))
        else:
            skipped += 1

    if not paired:
        print("[ERROR] Không tìm thấy cặp image-label nào!")
        sys.exit(1)

    print(f"  Images dir:    {IMAGES_DIR}")
    print(f"  Labels dir:    {LABELS_DIR}")
    print(f"  Paired:        {len(paired)}")
    if skipped > 0:
        print(f"  Skipped:       {skipped} (ảnh không có label)")
    print(f"  Split ratio:   train={TRAIN_RATIO} / valid={VALID_RATIO} / test={TEST_RATIO}")
    print(f"  Classes:       {CLASS_NAMES}")

    # --- Shuffle và split ---
    random.seed(SEED)
    random.shuffle(paired)

    n = len(paired)
    n_train = int(n * TRAIN_RATIO)
    n_valid = int(n * VALID_RATIO)
    n_test  = n - n_train - n_valid

    split_train = paired[:n_train]
    split_valid = paired[n_train:n_train + n_valid]
    split_test  = paired[n_train + n_valid:]

    print(f"\n  Train: {len(split_train)}")
    print(f"  Valid: {len(split_valid)}")
    print(f"  Test:  {len(split_test)}")

    # --- Tạo thư mục output ---
    splits = {
        "train": split_train,
        "valid": split_valid,
        "test":  split_test,
    }

    for split_name in splits:
        os.makedirs(os.path.join(OUTPUT_DIR, "images", split_name), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR, "labels", split_name), exist_ok=True)

    # --- Copy files ---
    print(f"\n[INFO] Copying files to {OUTPUT_DIR}/...\n")

    for split_name, file_pairs in splits.items():
        for img_name, label_name in file_pairs:
            src_img = os.path.join(IMAGES_DIR, img_name)
            src_lbl = os.path.join(LABELS_DIR, label_name)

            dst_img = os.path.join(OUTPUT_DIR, "images", split_name, img_name)
            dst_lbl = os.path.join(OUTPUT_DIR, "labels", split_name, label_name)

            shutil.copy2(src_img, dst_img)
            shutil.copy2(src_lbl, dst_lbl)

        print(f"  {split_name}: {len(file_pairs)} files copied")

    # --- Tạo data.yaml ---
    abs_output = os.path.abspath(OUTPUT_DIR)

    yaml_data = {
        "path":  abs_output,
        "train": "images/train",
        "val":   "images/valid",
        "test":  "images/test",
        "nc":    len(CLASS_NAMES),
        "names": CLASS_NAMES,
    }

    yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(yaml_data, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

    # --- Tổng kết ---
    print(f"\n{'=' * 60}")
    print(f"  DONE!")
    print(f"{'=' * 60}")
    print(f"  Output: {OUTPUT_DIR}/")
    print(f"     ├── images/")
    print(f"     │   ├── train/   ({len(split_train)} files)")
    print(f"     │   ├── valid/   ({len(split_valid)} files)")
    print(f"     │   └── test/    ({len(split_test)} files)")
    print(f"     ├── labels/")
    print(f"     │   ├── train/   ({len(split_train)} files)")
    print(f"     │   ├── valid/   ({len(split_valid)} files)")
    print(f"     │   └── test/    ({len(split_test)} files)")
    print(f"     └── data.yaml")
    print(f"\n  data.yaml content:")
    print(f"     path:  {abs_output}")
    print(f"     train: images/train")
    print(f"     val:   images/valid")
    print(f"     test:  images/test")
    print(f"     nc:    {len(CLASS_NAMES)}")
    print(f"     names: {CLASS_NAMES}")
    print(f"\n  Train YOLO:")
    print(f"     yolo detect train data={yaml_path} model=yolov8n.pt epochs=100 imgsz=640")
    print("=" * 60)


if __name__ == "__main__":
    main()

  DATASET SPLITTER (YOLO FORMAT)
  Images dir:    /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/output_sam3_0-5-conf-exclude-signal-10fps/frames_original
  Labels dir:    /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/output_sam3_0-5-conf-exclude-signal-10fps/labels_txt
  Paired:        603
  Split ratio:   train=0.7 / valid=0.2 / test=0.1
  Classes:       ['vehicle']

  Train: 422
  Valid: 120
  Test:  61

[INFO] Copying files to ./our-dataset-sam3-10fps-0-5-conf//...

  train: 422 files copied
  valid: 120 files copied
  test: 61 files copied

  DONE!
  Output: ./our-dataset-sam3-10fps-0-5-conf//
     ├── images/
     │   ├── train/   (422 files)
     │   ├── valid/   (120 files)
     │   └── test/    (61 files)
     ├── labels/
     │   ├── train/   (422 files)
     │   ├── valid/   (120 files)
     │   └── test/    (61 files)
     └── data.yaml

  data.yaml content:
     path:  /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_Ad

# YOLO training 

In [1]:
from ultralytics import YOLO

# Load a model
#model = YOLO("yolo26n.yaml")  # build a new model from YAML
#model = YOLO("yolo26n.pt")  # load a pretrained model (recommended for training)
model = YOLO("yolo26s.yaml").load("yolo26s.pt")  # build from YAML and transfer weights

# Train the model
results = model.train(data="./our-dataset-sam3-10fps-0-5-conf/data.yaml", epochs=30, imgsz=640)

Transferred 708/708 items from pretrained weights
Ultralytics 8.4.37 🚀 Python-3.12.0 torch-2.9.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 12282MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./our-dataset-sam3-10fps-0-5-conf/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=F

# YOLO detect result

In [3]:
! yolo detect predict model="/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/runs/detect/train/weights/best.pt" source='/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4' show_conf="False" show_labels="False"

WARNING ⚠️ argument 'show_conf=False,' does not require trailing comma ',', updating to 'show_conf=False'.
Ultralytics 8.4.37 🚀 Python-3.12.0 torch-2.9.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 12282MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs

video 1/1 (frame 1/6023) /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4: 384x640 28 vehicles, 28.8ms
video 1/1 (frame 2/6023) /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4: 384x640 27 vehicles, 4.8ms
video 1/1 (frame 3/6023) /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4: 384x640 27 vehicles, 4.6ms
video 1/1 (frame 4/6023) /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4: 384x640 29 vehicles, 4.9ms
video 1/1 (frame 5/6023) /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4: 

# Reduce predicted video file

In [12]:
! ffmpeg -i '/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/runs/detect/predict2/detection_output.avi' -vcodec libx265 -crf 32 '/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/runs/detect/predict2/detection_output_v2.mp4'


ffmpeg version 6.1.2 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 13.3.0 (conda-forge gcc 13.3.0-1)
  configuration: --prefix=/home/dmin/anaconda3/envs/ai --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1730671262093/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1730671262093/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=/home/conda/feedstock_root/build_artifacts/ffmpeg_1730671262093/_build_env/bin/x86_64-conda-linux-gnu-nm --ar=/home/conda/feedstock_root/build_artifacts/ffmpeg_1730671262093/_build_env/bin/x86_64-conda-linux-gnu-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --disable-gnutls --enable-libmp3lame --enable-libvpx --enable-libass --enable-pthreads --enable-vaapi --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libaom --enable-libsvta